# Text2SQL - Few-Shot Examples & Query Decomposition

1. **동적 Few-Shot** 예시 선택으로 SQL 생성 정확도 향상
2. 복잡한 질문의 **서브 쿼리 분해** 기법 학습
3. **벡터 스토어** 기반 예시 관리 시스템 구축

---

## 환경 설정 및 준비

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from pprint import pprint
import json

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
print(os.getenv('LANGSMITH_TRACING'))

`(3) SQLite DB`

In [ ]:
from langchain_community.utilities import SQLDatabase
import ast

db = SQLDatabase.from_uri("sqlite:///etf_database.db", ignore_tables=["ETFsInfo"])
print(db.dialect)
print(db.get_usable_table_names())
print(db.get_table_info()[:500])

`(4) 기본 컴포넌트 준비`

In [ ]:
from typing import TypedDict, Annotated, List
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import QuerySQLDatabaseTool
from langchain_core.vectorstores import InMemoryVectorStore
from langgraph.graph import START, END, StateGraph
from IPython.display import Image, display

# LLM 모델 생성
llm = ChatOpenAI(model="gpt-4.1-mini")

# 임베딩 모델
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

---

## 1. 기본 SQL QA Chain 

- 004에서 구현한 Entity Matching 포함 Chain을 기반으로 시작
- 고유명사 벡터 스토어 구축 포함

In [ ]:
import re

# 고유명사 추출
def query_as_list(db, query):
    res = db.run(query)
    res = [el for sub in ast.literal_eval(res) for el in sub if el]
    res = [re.sub(r"\b\d+\b", "", string).strip() for string in res]
    return list(set(res))

etfs = query_as_list(db, "SELECT DISTINCT 종목명 FROM ETFs")
fund_managers = query_as_list(db, "SELECT DISTINCT 운용사 FROM ETFs")
print(f"ETF 종목 수: {len(etfs)}, 운용사 수: {len(fund_managers)}")

# 엔티티 벡터 스토어
entity_store = InMemoryVectorStore(embeddings)
_ = entity_store.add_texts(etfs + fund_managers)
entity_retriever = entity_store.as_retriever(search_kwargs={"k": 10})

In [ ]:
from langchain_core.tools import tool

# 검색 도구 생성 - 하이브리드 검색기 사용
@tool("search_proper_nouns")
def entity_retriever_tool(query: str) -> str:
    """
    Use to look up values to filter on. Input is an approximate spelling 
    of the proper noun, output is valid proper nouns. Use the noun most 
    similar to the search.
    """
    docs = entity_retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

# 기본 State
class State(TypedDict):
    question: str
    query: str
    result: str
    answer: str

class QueryOutput(TypedDict):
    """Generated SQL query."""
    query: Annotated[str, ..., "Syntactically valid SQL query."]

# 기본 프롬프트 (엔티티 매칭 포함)
base_prompt_template = ChatPromptTemplate.from_template("""
Given an input question, create a syntactically correct {dialect} query to run to help find the answer.
Unless the user specifies a specific number, always limit to at most {top_k} results.
Order by relevant columns. Only ask for relevant columns.
Pay attention to column names in the schema. Be careful not to query non-existent columns.

Only use the following tables:
{table_info}

Entity names to consider:
{entity_info}

Question: {input}
""")

def write_query_basic(state: State):
    """기본 SQL 쿼리 생성 (엔티티 매칭)"""
    prompt = base_prompt_template.invoke({
        "dialect": db.dialect,
        "top_k": 10,
        "table_info": db.get_table_info(),
        "entity_info": entity_retriever_tool.invoke(state["question"]),
        "input": state["question"],
    })
    structured_llm = llm.with_structured_output(QueryOutput)
    result = structured_llm.invoke(prompt)
    return {"query": result["query"]}

def execute_query(state: State):
    """SQL 쿼리 실행"""
    try:
        tool = QuerySQLDatabaseTool(db=db)
        return {"result": tool.invoke(state["query"])}
    except Exception as e:
        return {"result": f"쿼리 실행 중 오류 발생: {str(e)}"}

def generate_answer(state: State):
    """답변 생성"""
    prompt = (
        "Given the following user question, corresponding SQL query, "
        "and SQL result, answer the user question.\n\n"
        f'Question: {state["question"]}\n'
        f'SQL Query: {state["query"]}\n'
        f'SQL Result: {state["result"]}'
    )
    response = llm.invoke(prompt)
    return {"answer": response.content}

# 기본 그래프
basic_graph = StateGraph(State)
basic_graph.add_node("write_query", write_query_basic)
basic_graph.add_node("execute_query", execute_query)
basic_graph.add_node("generate_answer", generate_answer)
basic_graph.add_edge(START, "write_query")
basic_graph.add_edge("write_query", "execute_query")
basic_graph.add_edge("execute_query", "generate_answer")
basic_chain = basic_graph.compile()

display(Image(basic_chain.get_graph().draw_mermaid_png()))

In [ ]:
# 기본 Chain 테스트
result = basic_chain.invoke({"question": "KB자산운용의 ETF 중 수익률 상위 5개는?"})
print(f"SQL: {result['query']}")
print(f"Answer: {result['answer']}")

---

## 2. Few-Shot Dynamic Examples

- **Few-Shot 학습**: 유사한 질문-SQL 쌍을 프롬프트에 포함하여 생성 정확도 향상
- **동적 선택**: 입력 질문과 가장 유사한 예시를 벡터 검색으로 자동 선택
- 참조: [OpenSearch-SQL](https://arxiv.org/pdf/2502.14913) - 동적 Few-Shot으로 +7% 정확도 향상

### 2.1 예시 데이터셋 구축

- ETF 데이터베이스에 맞는 다양한 유형의 Q&A 쌍 구축
- 난이도별, 유형별로 골고루 분포

In [ ]:
# Few-Shot 예시 데이터셋
examples = [
    # === 단순 조회 ===
    {
        "question": "ETF 총 개수는?",
        "sql": "SELECT COUNT(*) FROM ETFs",
        "category": "count"
    },
    {
        "question": "총보수가 0.1% 이하인 ETF는?",
        "sql": "SELECT 종목명, 총보수 FROM ETFs WHERE 총보수 <= 0.1 ORDER BY 총보수 ASC LIMIT 10",
        "category": "filter"
    },
    {
        "question": "순자산총액이 가장 큰 ETF는?",
        "sql": "SELECT 종목명, 순자산총액 FROM ETFs ORDER BY 순자산총액 DESC LIMIT 1",
        "category": "sort"
    },
    # === 집계 ===
    {
        "question": "운용사별 ETF 개수는?",
        "sql": "SELECT 운용사, COUNT(*) as cnt FROM ETFs GROUP BY 운용사 ORDER BY cnt DESC LIMIT 10",
        "category": "aggregation"
    },
    {
        "question": "평균 수익률이 가장 높은 분류체계는?",
        "sql": "SELECT 분류체계, AVG(수익률_최근1년) as avg_ret FROM ETFs GROUP BY 분류체계 ORDER BY avg_ret DESC LIMIT 5",
        "category": "aggregation"
    },
    {
        "question": "운용사별 평균 총보수를 비교해주세요",
        "sql": "SELECT 운용사, AVG(총보수) as avg_fee FROM ETFs GROUP BY 운용사 ORDER BY avg_fee ASC LIMIT 10",
        "category": "aggregation"
    },
    # === 조건 필터 ===
    {
        "question": "KB자산운용의 ETF 중 수익률 상위 5개는?",
        "sql": "SELECT 종목명, 수익률_최근1년 FROM ETFs WHERE 운용사 = 'KB자산운용' ORDER BY 수익률_최근1년 DESC LIMIT 5",
        "category": "filter_sort"
    },
    {
        "question": "삼성자산운용의 ETF 수는?",
        "sql": "SELECT COUNT(*) FROM ETFs WHERE 운용사 = '삼성자산운용'",
        "category": "filter_count"
    },
    # === JOIN (ETFs + ETFsInfo) ===
    {
        "question": "기초자산이 주식인 ETF의 평균 총보수는?",
        "sql": "SELECT AVG(e.총보수) as avg_fee FROM ETFs e JOIN ETFsInfo i ON e.종목코드 = i.종목코드 WHERE i.기초자산 = '주식'",
        "category": "join"
    },
    {
        "question": "기초자산이 채권인 ETF 목록은?",
        "sql": "SELECT e.종목명, e.총보수, e.수익률_최근1년 FROM ETFs e JOIN ETFsInfo i ON e.종목코드 = i.종목코드 WHERE i.기초자산 = '채권' LIMIT 10",
        "category": "join"
    },
    # === 복합 조건 ===
    {
        "question": "순자산 1000억 이상이고 총보수 0.3% 이하인 ETF는?",
        "sql": "SELECT 종목명, 순자산총액, 총보수 FROM ETFs WHERE 순자산총액 >= 100000 AND 총보수 <= 0.3 ORDER BY 순자산총액 DESC LIMIT 10",
        "category": "complex"
    },
    {
        "question": "수익률이 양수이고 변동성이 낮은 ETF를 추천해주세요",
        "sql": "SELECT 종목명, 수익률_최근1년, 변동성 FROM ETFs WHERE 수익률_최근1년 > 0 ORDER BY 변동성 ASC LIMIT 10",
        "category": "complex"
    },
    # === 비교 ===
    {
        "question": "분류체계별 ETF 개수를 비교해주세요",
        "sql": "SELECT 분류체계, COUNT(*) as cnt FROM ETFs GROUP BY 분류체계 ORDER BY cnt DESC",
        "category": "comparison"
    },
    {
        "question": "복제방법별 평균 추적오차는?",
        "sql": "SELECT 복제방법, AVG(추적오차) as avg_error FROM ETFs GROUP BY 복제방법 ORDER BY avg_error ASC",
        "category": "comparison"
    },
    # === 고급 ===
    {
        "question": "수익률 상위 10% ETF의 평균 총보수는?",
        "sql": "SELECT AVG(총보수) as avg_fee FROM (SELECT 총보수 FROM ETFs ORDER BY 수익률_최근1년 DESC LIMIT (SELECT COUNT(*) / 10 FROM ETFs))",
        "category": "advanced"
    },
]

print(f"예시 데이터셋: {len(examples)}개")
for ex in examples[:3]:
    print(f"  Q: {ex['question']}")
    print(f"  SQL: {ex['sql'][:80]}")
    print()

### 2.2 예시를 벡터 스토어에 저장

- 질문 텍스트를 임베딩하여 벡터 스토어에 저장
- 메타데이터로 SQL과 카테고리 정보 포함

In [ ]:
from langchain_core.documents import Document

# 예시를 Document로 변환하여 벡터 스토어에 저장
example_docs = [
    Document(
        page_content=ex["question"],
        metadata={"sql": ex["sql"], "category": ex["category"]}
    )
    for ex in examples
]

example_store = InMemoryVectorStore(embeddings)
_ = example_store.add_documents(example_docs)
example_retriever = example_store.as_retriever(search_kwargs={"k": 3})

print(f"예시 벡터 스토어 구축 완료: {len(example_docs)}개")

### 2.3 유사도 기반 예시 검색

- 입력 질문과 가장 유사한 예시 3개를 자동 검색
- 검색된 예시를 프롬프트에 포함

In [ ]:
# 예시 검색 테스트
test_query = "미래에셋자산운용의 ETF 수는 몇 개인가요?"
similar_examples = example_retriever.invoke(test_query)

print(f"입력 질문: {test_query}")
print(f"\n유사한 예시 {len(similar_examples)}개:")
for i, doc in enumerate(similar_examples, 1):
    print(f"  {i}. Q: {doc.page_content}")
    print(f"     SQL: {doc.metadata['sql']}")
    print(f"     Category: {doc.metadata['category']}")
    print()

In [ ]:
# 다른 유형 테스트
test_query2 = "기초자산이 원자재인 ETF의 수익률은?"
similar_examples2 = example_retriever.invoke(test_query2)

print(f"입력 질문: {test_query2}")
print(f"\n유사한 예시 {len(similar_examples2)}개:")
for i, doc in enumerate(similar_examples2, 1):
    print(f"  {i}. Q: {doc.page_content}")
    print(f"     SQL: {doc.metadata['sql']}")
    print()

### 2.4 프롬프트 템플릿에 동적 예시 삽입

- 검색된 예시를 프롬프트에 자동 삽입
- 기존 엔티티 매칭과 함께 사용

In [ ]:
def format_examples(docs):
    """검색된 예시를 프롬프트 형식으로 변환"""
    formatted = []
    for doc in docs:
        formatted.append(
            f"Question: {doc.page_content}\n"
            f"SQL: {doc.metadata['sql']}"
        )
    return "\n\n".join(formatted)

# Few-Shot 프롬프트 템플릿
fewshot_prompt_template = ChatPromptTemplate.from_template("""
Given an input question, create a syntactically correct {dialect} query to run to help find the answer.
Unless the user specifies a specific number, always limit to at most {top_k} results.
Order by relevant columns. Only ask for relevant columns.
Pay attention to column names in the schema.

Only use the following tables:
{table_info}

Entity names to consider:
{entity_info}

Here are some similar examples for reference:

{examples}

Now generate the SQL for this question:
Question: {input}
""")

def write_query_fewshot(state: State):
    """Few-Shot SQL 쿼리 생성"""
    # 유사 예시 검색
    similar_docs = example_retriever.invoke(state["question"])
    formatted = format_examples(similar_docs)
    
    prompt = fewshot_prompt_template.invoke({
        "dialect": db.dialect,
        "top_k": 10,
        "table_info": db.get_table_info(),
        "entity_info": entity_retriever_tool.invoke(state["question"]),
        "examples": formatted,
        "input": state["question"],
    })
    structured_llm = llm.with_structured_output(QueryOutput)
    result = structured_llm.invoke(prompt)
    return {"query": result["query"]}

# Few-Shot 그래프
fewshot_graph = StateGraph(State)
fewshot_graph.add_node("write_query", write_query_fewshot)
fewshot_graph.add_node("execute_query", execute_query)
fewshot_graph.add_node("generate_answer", generate_answer)
fewshot_graph.add_edge(START, "write_query")
fewshot_graph.add_edge("write_query", "execute_query")
fewshot_graph.add_edge("execute_query", "generate_answer")
fewshot_chain = fewshot_graph.compile()

display(Image(fewshot_chain.get_graph().draw_mermaid_png()))

### 2.5 Few-Shot 적용 전후 비교 테스트

In [ ]:
# Few-Shot 적용 전후 비교
test_questions = [
    "미래에셋자산운용에서 운용하는 ETF 중 총보수가 가장 낮은 3개는?",
    "기초자산이 채권인 ETF의 평균 수익률은?",
    "순자산 5000억 이상이고 수익률이 양수인 ETF는?",
    "분류체계별 평균 총보수를 비교해주세요",
    "수익률 상위 10개 ETF의 운용사 분포는?",
]

for q in test_questions:
    print("=" * 60)
    print(f"Q: {q}")
    
    # 기본 Chain
    basic_result = basic_chain.invoke({"question": q})
    print(f"\n[Basic]  SQL: {basic_result['query'][:100]}")
    
    # Few-Shot Chain
    fewshot_result = fewshot_chain.invoke({"question": q})
    print(f"[FewShot] SQL: {fewshot_result['query'][:100]}")
    print()
    print(f"[A-Basic]: {basic_result['answer']}")
    print("-" * 60)
    print(f"[A-FewShot]: {fewshot_result['answer']}")
    print()

---

## [실습] 나만의 예시 데이터셋 확장

### 문제
현재 예시 데이터셋(15개)에 5개 이상의 새로운 예시를 추가하고, 성능 변화를 측정하세요.

### 단계
1. ETF 데이터베이스의 스키마를 확인하고 새로운 유형의 질문-SQL 쌍 작성
2. 특히 기존 예시에 없는 패턴 추가 (HAVING, 서브쿼리, CASE WHEN 등)
3. 새 예시를 벡터 스토어에 추가
4. 테스트 질문으로 추가 전후 성능 비교

### 힌트
- `example_store.add_documents()` 메서드 활용
- HAVING, CASE WHEN 패턴 예시 추가 권장
- 추가 후 검색 결과 확인: `example_retriever.invoke("테스트 질문")`

In [ ]:
# 여기에 코드를 작성하세요.